In [0]:
%sql
select * from projectpractice2.silver.registered_users limit 10;

In [0]:
%sql
select * from projectpractice2.silver.user_info limit 10;

In [0]:
%sql
select * from projectpractice2.silver.bpm limit 10;

In [0]:
%sql
create table df7 as
select users.user_id, user_info.first_name, user_info.last_name, users.mac_address, user_info.state, bpm.device_id, 
cast(bpm.datetime as date) as date, round(avg(bpm.heartrate),2) as avg_heartrate,
avg(gym.ct_gym) as ct_gym, avg(gym.gym_duration_mins) as avg_gym_duration_mins from projectpractice2.silver.bpm as bpm
left join projectpractice2.silver.registered_users as users on bpm.device_id = users.device_id
left join projectpractice2.silver.user_info as user_info on user_info.user_id = users.user_id
left join projectpractice2.silver.date_lookup_table as dt on dt.date = cast(bpm.datetime as date)
left join projectpractice2.silver.gym_logins as gym on gym.mac_address = users.mac_address
group by users.user_id, user_info.first_name, user_info.last_name, users.mac_address, user_info.state, bpm.device_id
, cast(bpm.datetime as date)
order by users.user_id, cast(bpm.datetime as date);

In [0]:
%sql
create table if not exists projectpractice2.gold.gym_agg;

In [0]:
df7 = spark.table("df7")

(
    df7.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("projectpractice2.gold.gym_agg")
)

In [0]:
%sql
select * from projectpractice2.gold.gym_agg;